In [1]:
import requests
from pydantic import BaseModel

In [2]:
class SearchResult(BaseModel):
    page_id: int
    title: str
    snippet: str


class WikipediaPage(BaseModel):
    page_id: int
    title: str
    url: str
    content: str


In [3]:
BASE_URL = "https://en.wikipedia.org/w/api.php"
headers = {"User-Agent": "QuizApp/0.1 (abc@example.com)"}


def search(query: str, limit: int = 5) -> list[SearchResult]:
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": limit,
    }

    response = requests.get(BASE_URL, headers=headers, params=params)
    print(response)
    response.raise_for_status()

    data = response.json()["query"]["search"]

    return [
        SearchResult(
            page_id=result["pageid"],
            title=result["title"],
            snippet=result["snippet"],
        )
        for result in data
    ]

def get_page(title: str) -> WikipediaPage:
    params = {
        "action": "query",
        "prop": "extracts",
        "titles": title,
        "explaintext": True,
        "format": "json",
        "exsectionformat": "wiki",
        "redirects": 1
    }

    response = requests.get(BASE_URL, headers=headers, params=params)

    pages = response.json()["query"]["pages"]

    page = next(iter(pages.values()))

    return WikipediaPage(
        page_id=page["pageid"],
        title=page["title"],
        url=f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}",
        content=page.get("extract", "")
    )

def resolve_topic_to_article(topic):
    results = search(topic, limit=3)
    if not results:
        return None
    top = results[0]
    content = get_page(top.title)
    return {"title": top.title, "content": content.content}

In [4]:
topic = "Linux"
result = resolve_topic_to_article(topic=topic)

<Response [200]>


In [5]:
import re

BOILERPLATE_SECTIONS = {
    "see also",
    "references",
    "external links",
    "further reading",
    "notes",
    "citations",
    "bibliography",
    "sources",
}


def parse_sections(full_text):
    heading_pattern = re.compile(r"^(=+)\s*(.*?)\s*=+$")
    lines = full_text.split("\n")

    raw_sections = []
    current = {"title": None, "level": 2, "text_lines": []}

    for line in lines:
        match = heading_pattern.match(line.strip())
        if match:
            raw_sections.append(current)
            level = len(match.group(1))
            current = {
                "title": match.group(2).strip(),
                "level": level,
                "text_lines": [],
            }
        else:
            current["text_lines"].append(line)
    raw_sections.append(current)

    stack = []
    sections = []
    for s in raw_sections:
        title = s["title"] or "Introduction"
        if title.lower() in BOILERPLATE_SECTIONS:
            continue

        while stack and stack[-1][0] >= s["level"]:
            stack.pop()
        stack.append((s["level"], title))

        text = "\n".join(l for l in s["text_lines"] if l.strip())
        if not text:
            continue

        sections.append(
            {
                "title": title,  # leaf title — new
                "breadcrumb": " > ".join(t for _, t in stack),
                "text": text,
            }
        )

    return sections


In [6]:
from urllib.parse import quote


def wiki_section_url(article_title, section_title):
    base = f"https://en.wikipedia.org/wiki/{quote(article_title.replace(' ', '_'))}"
    if section_title == "Introduction":
        return base  # intro has no heading, so no anchor to link to
    anchor = quote(section_title.replace(" ", "_"))
    return f"{base}#{anchor}"


In [7]:
sections = parse_sections(result.get("content", ""))
sections

[{'title': 'Introduction',
  'breadcrumb': 'Introduction',
  'text': 'Linux ( LIN-uuks) is a family of free and open-source software Unix-like operating systems based on the Linux kernel, which was first released on 17 September 1991 by Linus Torvalds. Some members of the family are typically packaged as a distribution (a.k.a. distro), which includes the kernel alongside supporting system software and libraries developed by third parties—such as GNU, Red Hat, and X.Org—to create a complete operating system; however, not all Linux-based operating systems are considered distros, with Android being an example. Linux was originally designed as a clone of Unix and is distributed under the copyleft GPL license.\nThere are many thousands of Linux distributions, many based directly or indirectly on other distributions; popular Linux distros include Debian, Fedora Linux, Linux Mint, Arch Linux, and Ubuntu, while commercial distributions include Red Hat Enterprise Linux, SUSE Linux Enterprise, a

In [8]:
import tiktoken

encoder = tiktoken.get_encoding("cl100k_base")


def count_tokens(text):
    return len(encoder.encode(text))


MIN_TOKENS = 50
MAX_TOKENS = 400


def build_chunks(sections, article_title):
    chunks = []
    buffer = None

    for section in sections:
        tokens = count_tokens(section["text"])

        if tokens < MIN_TOKENS:
            if buffer is None:
                buffer = section
            else:
                buffer["text"] += "\n\n" + section["text"]
            continue

        if buffer is not None:
            chunks.extend(split_if_needed(buffer, article_title))
            buffer = None
        chunks.extend(split_if_needed(section, article_title))

    if buffer is not None:
        chunks.extend(split_if_needed(buffer, article_title))

    return chunks


def split_if_needed(section, article_title):
    text = section["text"]
    if count_tokens(text) <= MAX_TOKENS:
        return [format_chunk(article_title,section["title"], section["breadcrumb"], text)]

    paragraphs = [p for p in text.split("\n\n") if p.strip()]
    sub_chunks, current, current_tokens = [], [], 0

    for para in paragraphs:
        para_tokens = count_tokens(para)
        if current and current_tokens + para_tokens > MAX_TOKENS:
            sub_chunks.append("\n\n".join(current))
            current = [current[-1], para]  # 1-paragraph overlap
            current_tokens = count_tokens(current[0]) + para_tokens
        else:
            current.append(para)
            current_tokens += para_tokens

    if current:
        sub_chunks.append("\n\n".join(current))

    return [format_chunk(article_title, section["title"], section["breadcrumb"], sc) for sc in sub_chunks]


def format_chunk(article_title, section_title, breadcrumb, text):
    return {
        "text": f"Article: {article_title}\nSection: {breadcrumb}\n\n{text}",
        "article_title": article_title,
        "section_title": section_title,
        "section_breadcrumb": breadcrumb,
        "source_url": wiki_section_url(article_title, section_title),
        "raw_text": text,
    }


In [9]:
chunks = build_chunks(sections=sections, article_title=result.get("title", ""))

In [10]:
chunks[2]

{'text': 'Article: Linux\nSection: History > Precursors\n\nThe Unix operating system was conceived of and implemented in 1969, at AT&T\'s Bell Labs in the United States, by Ken Thompson, Dennis Ritchie, Douglas McIlroy, and Joe Ossanna. First released in 1971, Unix was written entirely in assembly language, as was common practice at the time. In 1973, in a key pioneering approach, it was rewritten in the C programming language by Dennis Ritchie (except for some hardware and I/O routines). The availability of a high-level language implementation of Unix made its porting to different computer platforms easier.\nAs a 1956 antitrust case forbade AT&T from entering the computer business, AT&T provided the operating system\'s source code to anyone who asked. As a result, Unix use grew quickly and it became widely adopted by academic institutions and businesses. In 1984, AT&T divested itself of its regional operating companies, and was released from its obligation not to enter the computer bu

## Embeddings

In [11]:
from google import genai
from google.genai import types
import os

In [12]:
gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [13]:
def get_embedding(text, emb_model="gemini-embedding-2"):
    result = gemini_client.models.embed_content(
        model=emb_model,
        contents=text,
        config=types.EmbedContentConfig(output_dimensionality=1536),
    )


    return result.embeddings[0].values

In [14]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [15]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [16]:
qdrant_client.delete_collection(collection_name="Quiz-App-Dev-Collection")

False

In [16]:
qdrant_client.create_collection(
    collection_name="Quiz-App-Dev-Collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)


True

In [17]:
data_to_embed = chunks

In [18]:
pointstructs = []
for i, data in enumerate(data_to_embed):
    embedding = get_embedding(data["text"])
    pointstructs.append(PointStruct(id=i, vector=embedding, payload={
        "article_title": data["article_title"],
        "section_title": data["section_title"],
        "section_breadcrumb": data["section_breadcrumb"],
        "source_url": data["source_url"],
        "raw_text": data["raw_text"]
    }))


In [19]:
qdrant_client.upsert(
    collection_name="Quiz-App-Dev-Collection", wait=True, points=pointstructs
)


UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [20]:
def build_outline(sections, article_title):
    outline = []
    for s in sections:
        preview = s["raw_text"][:150].rsplit(" ", 1)[0] + "..."
        outline.append(
            {
                "breadcrumb": s["section_breadcrumb"],
                "preview": preview,
                "token_count": count_tokens(s["raw_text"]),
            }
        )
    return {"article_title": article_title, "sections": outline}


In [21]:
outline = build_outline(chunks, topic)

In [22]:
outline

{'article_title': 'Linux',
 'sections': [{'breadcrumb': 'Introduction',
   'preview': 'Linux ( LIN-uuks) is a family of free and open-source software Unix-like operating systems based on the Linux kernel, which was first released on 17...',
   'token_count': 369},
  {'breadcrumb': 'Overview',
   'preview': 'The Linux kernel was created by Linus Torvalds, following the lack of a working kernel for GNU, a Unix-compatible operating system made entirely of...',
   'token_count': 590},
  {'breadcrumb': 'History > Precursors',
   'preview': "The Unix operating system was conceived of and implemented in 1969, at AT&T's Bell Labs in the United States, by Ken Thompson, Dennis Ritchie,...",
   'token_count': 537},
  {'breadcrumb': 'History > Creation',
   'preview': 'While attending the University of Helsinki in the fall of 1990, Torvalds enrolled in a Unix course. The course used a MicroVAX minicomputer running...',
   'token_count': 442},
  {'breadcrumb': 'History > Copyright, trademark, and n

In [23]:
len(str(build_outline(chunks, topic)).split(" ")) 

559

In [24]:
import json

In [25]:
import sqlite3
from datetime import datetime, timezone


def slugify_title(title):
    return re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")


def get_connection(db_path="../quiz_outlines.db"):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def create_outline_table(conn):
    conn.execute("""
        CREATE TABLE IF NOT EXISTS outlines (
            article_id    TEXT PRIMARY KEY,
            article_title TEXT UNIQUE NOT NULL,
            outline_json  TEXT NOT NULL,
            fetched_at    TEXT NOT NULL
        )
    """)
    conn.commit()


In [26]:
def save_outline(conn, article_title, outline_dict):
    article_id = slugify_title(article_title)
    conn.execute(
        """
        INSERT INTO outlines (article_id, article_title, outline_json, fetched_at)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(article_id) DO UPDATE SET
            outline_json = excluded.outline_json,
            fetched_at   = excluded.fetched_at
    """,
        (
            article_id,
            article_title,
            json.dumps(outline_dict),
            datetime.now(timezone.utc).isoformat(),
        ),
    )
    conn.commit()
    return article_id


In [27]:
def get_outline(conn, article_title):
    row = conn.execute(
        "SELECT article_id, outline_json, fetched_at FROM outlines WHERE article_title = ?",
        (article_title,),
    ).fetchone()
    if row is None:
        return None
    return {
        "article_id": row["article_id"],
        "outline": json.loads(row["outline_json"]),
        "fetched_at": row["fetched_at"],
    }


In [28]:
conn = get_connection()
create_outline_table(conn)

outline = build_outline(chunks, topic)
article_id = save_outline(conn, topic, outline)


In [29]:
cached = get_outline(conn, topic)
cached

{'article_id': 'linux',
 'outline': {'article_title': 'Linux',
  'sections': [{'breadcrumb': 'Introduction',
    'preview': 'Linux ( LIN-uuks) is a family of free and open-source software Unix-like operating systems based on the Linux kernel, which was first released on 17...',
    'token_count': 369},
   {'breadcrumb': 'Overview',
    'preview': 'The Linux kernel was created by Linus Torvalds, following the lack of a working kernel for GNU, a Unix-compatible operating system made entirely of...',
    'token_count': 590},
   {'breadcrumb': 'History > Precursors',
    'preview': "The Unix operating system was conceived of and implemented in 1969, at AT&T's Bell Labs in the United States, by Ken Thompson, Dennis Ritchie,...",
    'token_count': 537},
   {'breadcrumb': 'History > Creation',
    'preview': 'While attending the University of Helsinki in the fall of 1990, Torvalds enrolled in a Unix course. The course used a MicroVAX minicomputer running...',
    'token_count': 442},
   {'br